# Commodity prediction · Manual research readiness

**Milestone:** verified dependencies, preserved controls, and a training-only feature audit.

This notebook does **not** train models, load saved models, score the final test, or rerun
notebooks 00–02. Saved development scores are historical evidence, not new results.
The final 247 origins remain reserved. Feature engineering remains open.

Run this notebook using **Commodity - manual (verified)**. The helper reuses a matching
locked environment or creates an independent one; it never changes an older environment.


## Publication provenance

This is a publication copy, not a new execution. Original code cells: 9; cells with execution counts: 9; original error outputs: 0; inline Plotly outputs retained: 5. Console logs, table/HTML output, attachments and tracebacks were omitted from this copy. The original remains in the research workspace. An execution count alone is not proof of success. Refer to the linked result ledger for completed/failed/pending study status. Interactive figure data are disclosed with this public notebook.


In [1]:
from pathlib import Path
import os, sys, json, importlib.util
from IPython.display import display, Markdown
import pandas as pd
import plotly.graph_objects as go

ROOT = Path(os.environ.get('COMMODITY_MANUAL_PROJECT', '/home/sagemaker-user/projects/commodity-prediction-manual'))
HELPER = Path(os.environ.get('COMMODITY_MANUAL_HELPER', '/home/sagemaker-user/commodity_prepare_manual.py'))
spec = importlib.util.spec_from_file_location('manual_support', HELPER)
support = importlib.util.module_from_spec(spec)
spec.loader.exec_module(support)
receipt = support.check_notebook_ready(ROOT)
print('Verified source:', receipt['source_commit'])
print('Environment:', receipt['runtime']['mode'])
print('Model fits in this milestone: 0')


## 1 · What is ready?

Five sealed parent stages are sufficient for the pending normalization experiment:
one reusable feature panel, one study summary, and three saved `current_market` models.
Hash verification is not prediction replay. The experiment must replay each control
before fitting. Raw-data checks and this notebook are not an off-disk backup.


In [2]:
display(pd.DataFrame([
    ['Source revision', receipt['source_commit']],
    ['Raw data files verified', receipt['raw_files_verified']],
    ['Recovered parent stages', len(receipt['recovered_stages'])],
    ['Focused feature tests passed', receipt['focused_tests']['passed']],
    ['Locked Python interpreter', receipt['runtime']['python']],
    ['Model loads / fits', '0 / 0'],
    ['Final-test evaluations', 0],
], columns=['Check', 'Verified result']))
figures = []
def show(fig, slug):
    fig.update_layout(height=max(fig.layout.height or 0, 460), margin=dict(l=70,r=35,t=80,b=80),
                      font=dict(size=14), hovermode='closest')
    fig.show(renderer="plotly_mimetype")
    figures.append((slug, fig))


## 2 · Saved development results

The following chart reads the recovered, sealed market-path summary. It does not
rerun an experiment. The observations and metric must match before comparisons
are made; no leaderboard score is plotted on these axes.


In [3]:
scores = pd.DataFrame([
    {'Representation':name, 'Score':s['official_metric']}
    for name,s in receipt['saved_scores'].items()
]).sort_values('Score')
fig = go.Figure(go.Bar(x=scores.Score.tolist(), y=scores.Representation.tolist(),
                      orientation='h', text=[f'{v:.6f}' for v in scores.Score],
                      textposition='outside', cliponaxis=False))
fig.update_layout(title='Saved development score · higher is better',
                  xaxis_title='Mean daily rank correlation / population standard deviation',
                  yaxis_title='Representation')
show(fig, 'saved_development_scores')


## 3 · Time-period stability, not just a pooled number

A useful representation should improve the relevant periods, not just one pooled
point estimate. The weak middle period is visible here. These fold values are
also read from saved evidence; there is no new validation selection.


In [4]:
fig = go.Figure()
for name in ['historical_mean','admitted_tail','current_market','price_path','activity_path','joint_path']:
    s = receipt['saved_scores'].get(name)
    if s and len(s['fold_scores']) == 3:
        fig.add_trace(go.Scatter(x=['Fold 1','Fold 2','Fold 3'], y=s['fold_scores'],
                                 name=name, mode='lines+markers'))
fig.update_layout(title='Saved development performance by chronological fold',
                  xaxis_title='Purged validation period', yaxis_title='Official metric',
                  hovermode='x unified')
show(fig, 'saved_fold_stability')


## 4 · Inspect the new features on training inputs only

This bounded audit reads only the first **1,164 training market rows** and target
metadata. It does not open training labels, validation outcomes or held-out rows.
It constructs the 14-template joint normalization block already implemented in
GitHub, then reports availability and numerical distributions. Structural absence
is counted separately from missing observations; histograms exclude structurally
inapplicable pair entries. **Coverage and
numerical sanity do not prove predictive value.** The declared fitted ablation
is still required.


In [5]:
support.audit_live(ROOT, HELPER, receipt)
audit = support.json_read(ROOT / 'logs/manual_readiness/feature_audit.json')
print('Audit status:', audit['status'])
print('Training rows / targets / templates:', audit['rows'], audit['targets'], audit['templates'])
print('New fits:', audit['new_training_fits'])
feature_table = pd.DataFrame(audit['feature_statistics'])
display(feature_table[['name','applicable_fraction','finite_within_applicable','q05','median','q95']])


In [6]:
fig = go.Figure()
fig.add_trace(go.Bar(x=(100*feature_table.applicable_fraction).tolist(),
    y=feature_table.name.tolist(), orientation='h', name='Structurally applicable (% of all entries)'))
fig.add_trace(go.Bar(x=(100*feature_table.finite_within_applicable).tolist(),
    y=feature_table.name.tolist(), orientation='h', name='Finite (% of applicable entries)'))
fig.update_layout(title='Feature availability · structural zeros are not observed signals',
                  barmode='group', xaxis_title='Percentage (see each denominator in legend)', yaxis_title='Candidate template',
                  yaxis=dict(automargin=True), height=600)
fig.update_xaxes(range=[0,100])
show(fig, 'normalization_training_coverage')


In [7]:
fig = go.Figure()
for row in audit['feature_statistics']:
    if 'applicable_legs' in row['name']:
        continue
    h = row['histogram']
    if h['finite_count']:
        centers = [(a+b)/2 for a,b in zip(h['edges'][:-1],h['edges'][1:])]
        fig.add_trace(go.Scatter(x=centers,
            y=[v/h['finite_count'] for v in h['counts']],
            mode='lines', name=row['name'].removeprefix('market_normalization__')))
fig.update_layout(title='Training-only distributions · applicable pairs only',
                  xaxis_title='Transformed feature value', yaxis_title='Fraction of finite entries')
show(fig, 'normalization_training_distributions')


## 5 · The next experiment is declared, not completed

Test `normalized_price`, `volume_confirmation`, and their exact union against
saved `current_market` controls. Three first-fold fits earn continuation only if
at least one gain reaches **0.002**, followed by inspection. At most nine new fits
and 300 cumulative study seconds are declared. No automatic full run occurs here.
No family is retained just because its chart looks plausible.


In [8]:
fig = go.Figure(go.Bar(x=['Normalized price','Volume confirmation','Exact union'],
                      y=[7,7,14], text=[7,7,14], textposition='outside'))
fig.update_layout(title='Pending ablation panels · added templates, not proven signals',
                  xaxis_title='Feature family', yaxis_title='Added candidate templates')
show(fig, 'declared_normalization_panels')


## 6 · Save the compact receipt and offline interactive dashboard

The HTML export is self-contained and uses Plotly only. It needs neither Chrome
nor Kaleido nor Jinja2. Keep the audit local until its contents are reviewed for
publication. Existing canonical notebooks remain unchanged. Save this notebook
with Ctrl+S, then stop the SageMaker space after downloading the receipt.


In [9]:
out = support.save_dashboard(ROOT, receipt, audit, figures)
display(Markdown('**Milestone complete: dependencies and training-only feature audit verified.**'))
print('RESULT: NOTEBOOK_AND_FEATURE_AUDIT_READY')
print('REPORT:', out['report'])
print('DASHBOARD:', out['dashboard'])
print('NEXT: Save this notebook, download the small JSON report, and stop the space.')
print('No first-fold model fits were launched. No previous model or notebook was rerun.')
